# Data Cleaning

## Orders Table

Problems to investigate
 - Missing values at 
   - Order_approved_at
   - Order_delivered_carrier_date
   - Order_delivered_customer_date
 - Convert dates to datetime 
 - Invalid dates 
   - Are there delivery dates scheduled before order approved, etc.
   - Are there timestamps outside the dataset's date range or dates that occur before the purchase timestamp?
   - Do the timestamps make sense for each order status?



In [1]:
import pandas as pd

orders_raw = pd.read_csv('../data/raw/olist_orders_dataset.csv')
orders = orders_raw.copy()

print("Original Shape:", orders_raw.shape)
print("Working Shape:", orders.shape)

Original Shape: (99441, 8)
Working Shape: (99441, 8)


In [5]:
# Convert timestamps to datetime
dates = ["order_purchase_timestamp",
         "order_approved_at",
         "order_delivered_carrier_date",
         "order_delivered_customer_date",
         "order_estimated_delivery_date"]

for field in dates:
    orders[field] = pd.to_datetime(orders[field], errors="coerce")

print("Check conversions:\n", orders[dates].dtypes)
print("\nCheck null vals:\n", orders[dates].isnull().sum())

Check conversions:
 order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Check null vals:
 order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [7]:
# Investigate Missing Vals
missing_by_status = orders.groupby("order_status")[dates].apply(
    lambda group: group.isnull().sum()
)

missing_by_status

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0,0,2,2,0
canceled,0,141,550,619,0
created,0,5,5,5,0
delivered,0,14,2,8,0
invoiced,0,0,314,314,0
processing,0,0,301,301,0
shipped,0,0,0,1107,0
unavailable,0,0,609,609,0


In [67]:
#Investigate Delivered Orders with Missing Approval

delivered_missing_approval = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].isna())
]

delivered_missing_approval[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]


,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38
22663,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23
48401,7002a78c79c519ac54022d4f8a65e6e8,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19
61743,2eecb0d85f281280f79fa00f9cec1a95,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03


In [70]:
# Do all delivered orders missing approval have recorded carrier and customer deliveries
print(delivered_missing_approval[
    delivered_missing_approval["order_delivered_carrier_date"].isna() |
    delivered_missing_approval["order_delivered_customer_date"].isna()])

Empty DataFrame
Columns: [order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]
Index: []


In [72]:
# Do all delivered orders missing approval have correct chronological dates
chronological_check = delivered_missing_approval[
    (delivered_missing_approval["order_purchase_timestamp"] < 
    delivered_missing_approval["order_delivered_carrier_date"]) &
    (delivered_missing_approval["order_delivered_carrier_date"] <
     delivered_missing_approval["order_delivered_customer_date"])
    ]

print("Do all delivered orders missing approval have correct chronological dates:", len(chronological_check) == len(delivered_missing_approval))


Do all delivered orders missing approval have correct chronological dates: True


In [68]:
# Dates of orders delivered missing approval
delivered_missing_approval[
    "order_purchase_timestamp"
].dt.to_period("M").value_counts().sort_index()

order_purchase_timestamp
2017-01     2
2017-02    12
Freq: M, Name: count, dtype: int64

### Missing Approval Timestamps in Delivered Orders

- Identified 14 delivered orders with missing approval timestamps.
- All 14 orders had recorded carrier handoff and customer delivery timestamps.
- The remaining available timestamps followed the expected chronological sequence.
- The actual approval timestamps cannot be determined from the available data.
- These records will be retained, with missing approval timestamps preserved as NaT.
- Approval-related duration calculations will exclude these records.

In [73]:
# Investigate Missing Customer Delivery Dates in Delivered Orders

delivered_missing_customer_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_missing_customer_date[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


### Missing Customer Delivery Timestamps in Delivered Orders

- Identified 8 delivered orders with missing customer delivery timestamps.
- 7 orders had recorded carrier handoff timestamps.
- 1 order was missing both carrier handoff and customer delivery timestamps.
- All 8 orders had purchase and approval timestamps.
- These records will be retained without imputing delivery dates.
- Records will be excluded from delivery-duration calculations requiring actual customer delivery timestamps.

In [74]:
# Investigate Delivered Orders Missing Carrier Handoff

delivered_missing_carrier = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isna())
]

print("Delivered orders missing carrier handoff:",
      len(delivered_missing_carrier))

delivered_missing_carrier[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

Delivered orders missing carrier handoff: 2


,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


### Missing Carrier Handoff Timestamps in Delivered Orders

- Identified 2 delivered orders with missing carrier handoff timestamps.
- Order 73222 had a recorded customer delivery timestamp but no carrier handoff timestamp.
- Order 92643 was missing both carrier handoff and customer delivery timestamps.
- Both orders had recorded purchase and approval timestamps.
- Both records will be retained without imputing missing dates.
- These records will be excluded from calculations requiring a valid carrier handoff timestamp.
- Order 73222 can still contribute to late-delivery analysis because its customer delivery timestamp is available.
- Order 92643 cannot be classified as late or on-time based on the available actual delivery timestamp.

In [16]:
# Investigate possible invalid dates

# Does approval happen before purchase
invalid_approval = orders[
    orders["order_approved_at"].notna() 
    & (orders["order_approved_at"] < orders["order_purchase_timestamp"])]

print("Invalid Approvals:", len(invalid_approval))

# Are all order carrier handoffs after approval
invalid_carrier = orders[
    orders["order_delivered_carrier_date"].notna() 
    & orders["order_approved_at"].notna()
    & (orders["order_delivered_carrier_date"] < orders["order_approved_at"])]

print("\nInvalid Carrier Delivery Dates:", len(invalid_carrier))

# Are all order customer delivery dates after carrier handoffs
invalid_cust_delivery = orders[
    orders["order_delivered_carrier_date"].notna() 
    & orders["order_delivered_customer_date"].notna()
    & (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])]

print("\nInvalid Customer Delivery Dates:", len(invalid_cust_delivery))

# Are all estimated customer delivery dates after carrier handoffs
invalid_estimate = orders[
    orders["order_delivered_carrier_date"].notna() 
    & orders["order_estimated_delivery_date"].notna()
    & (orders["order_estimated_delivery_date"] < orders["order_delivered_carrier_date"])]

print("\nInvalid Delivery Estimates:", len(invalid_estimate))

Invalid Approvals: 0

Invalid Carrier Delivery Dates: 1359

Invalid Customer Delivery Dates: 23

Invalid Delivery Estimates: 473


In [34]:
# Inspect Invalid Carrier Dates

invalid_carrier[
     [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date"
    ]
].head(20)

print(invalid_carrier["order_status"].value_counts())

order_status
delivered    1350
shipped         9
Name: count, dtype: int64


In [35]:
# Calculate how much earlier carrier handoff occurred than approval
invalid_carrier = invalid_carrier.copy()

invalid_carrier["hours_before_approval"] = (
    invalid_carrier["order_approved_at"] -
    invalid_carrier["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

invalid_carrier["hours_before_approval"].describe()

count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
max      4109.256111
Name: hours_before_approval, dtype: float64

In [36]:
invalid_carrier["approval_month"] = (
    invalid_carrier["order_approved_at"].dt.to_period("M")
)

invalid_carrier["approval_month"].value_counts().sort_index()

approval_month
2017-03      1
2017-04     24
2017-05     14
2017-06      2
2017-07      3
2017-08     21
2017-09     13
2017-12      6
2018-01      1
2018-02     36
2018-03      2
2018-04    393
2018-05     68
2018-06     95
2018-07    565
2018-08    115
Freq: M, Name: count, dtype: int64

In [37]:
# Total orders approved in each month
monthly_total = (
    orders[orders["order_approved_at"].notna()]
    .groupby(orders["order_approved_at"].dt.to_period("M"))
    .size()
)

# Number of carrier-before-approval anomalies per month
monthly_anomalies = (
    invalid_carrier.groupby("approval_month").size()
)

# Combine both results
monthly_comparison = pd.DataFrame({
    "total_orders": monthly_total,
    "invalid_carrier_dates": monthly_anomalies
}).fillna(0)

# Calculate anomaly percentage
monthly_comparison["anomaly_percentage"] = (
    monthly_comparison["invalid_carrier_dates"] /
    monthly_comparison["total_orders"] * 100
)

monthly_comparison.round(2)

,total_orders,invalid_carrier_dates,anomaly_percentage
2016-09,1,0.0,0.00
2016-10,320,0.0,0.00
2016-12,1,0.0,0.00
2017-01,760,0.0,0.00
2017-02,1765,0.0,0.00
2017-03,2689,1.0,0.04
2017-04,2374,24.0,1.01
2017-05,3693,14.0,0.38
2017-06,3252,2.0,0.06
2017-07,3974,3.0,0.08


### Carrier Timestamp Anomalies

- Identified 1,359 orders where carrier handoff occurred before order approval.
- 1,350 were marked delivered and 9 were marked shipped.
- The average discrepancy was approximately 24.75 hours.
- The median discrepancy was approximately 17.17 hours.
- April and July 2018 accounted for approximately 70.5% of these records.
- Records will be retained but flagged as having inconsistent timestamps.
- These records should be excluded from analyses requiring a valid approval-to-carrier timeline.

In [39]:
# Inspect Invalid Customer Deliveries
invalid_cust_delivery[
    [
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

print(invalid_cust_delivery["order_status"].value_counts())

order_status
delivered    23
Name: count, dtype: int64


In [40]:
invalid_cust_delivery = invalid_cust_delivery.copy()

invalid_cust_delivery["hours_before_carrier"] = (
    invalid_cust_delivery["order_delivered_carrier_date"] -
    invalid_cust_delivery["order_delivered_customer_date"]
).dt.total_seconds() / 3600

invalid_cust_delivery["hours_before_carrier"].describe()

count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: hours_before_carrier, dtype: float64

### Customer Delivery Timestamp Anomalies

- Identified 23 orders where customer delivery occurred before carrier handoff.
- All 23 orders were marked as delivered.
- The average discrepancy was approximately 78.46 hours.
- The median discrepancy was approximately 39.86 hours.
- These records represent chronological inconsistencies.
- Records will be retained but flagged for data-quality purposes.
- Affected timestamps will be excluded from calculations requiring a valid carrier-to-customer delivery timeline.

In [30]:
# Delivered Orders with Missing Timestamps

delivered_missing_dates = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isnull() |
        orders["order_delivered_carrier_date"].isnull() |
        orders["order_delivered_customer_date"].isnull()
    )
]

print("Delivered orders with missing timestamps:", len(delivered_missing_dates))

delivered_missing_dates.head(20)

Delivered orders with missing timestamps: 23


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22


In [17]:
# Late Deliveries (Orders delivered after estimated delivery date)

late_deliveries = orders[
    orders["order_delivered_customer_date"].notna() &
    orders["order_estimated_delivery_date"].notna() &
    (orders["order_estimated_delivery_date"] < orders["order_delivered_customer_date"])]

print("Orders delivered after estimated date:", len(late_deliveries))

Orders delivered after estimated date: 7827


In [44]:
# Investigate Invalid Delivery Estimates

invalid_estimate[
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

print(invalid_estimate["order_status"].value_counts())

order_status
delivered    465
shipped        8
Name: count, dtype: int64


In [58]:
late_deliv_estimate = invalid_estimate[
    invalid_estimate["order_delivered_customer_date"].notna() & 
    invalid_estimate["order_estimated_delivery_date"] .notna() &
    (invalid_estimate["order_delivered_customer_date"] > invalid_estimate["order_estimated_delivery_date"])
]

print("Late Deliveries in Invalid Estimates:", len(late_deliv_estimate))
print("\nTotal Invalid Estimates:", len(invalid_estimate))

Late Deliveries in Invalid Estimates: 465

Total Invalid Estimates: 473


In [59]:
not_late_estimates = invalid_estimate[
    invalid_estimate["order_delivered_customer_date"].notna() &
    (
        invalid_estimate["order_delivered_customer_date"] <=
        invalid_estimate["order_estimated_delivery_date"]
    )
]

not_late_estimates[
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [64]:
print(
    "Customer delivery timestamps missing:",
    invalid_estimate["order_delivered_customer_date"].isna().sum()
)

missing_deliv = invalid_estimate[invalid_estimate["order_delivered_customer_date"].isna()]
print("\nCustomer deliivery timestamp missing statuses:\n", missing_deliv["order_status"].value_counts())

print(
    "Customer delivery timestamps present:",
    invalid_estimate["order_delivered_customer_date"].notna().sum()
)

Customer delivery timestamps missing: 8

Customer deliivery timestamp missing statuses:
 order_status
shipped    8
Name: count, dtype: int64
Customer delivery timestamps present: 465


In [61]:
print("Late deliveries:", len(late_deliv_estimate))

print(
    "Delivered on or before estimated date:",
    (
        invalid_estimate["order_delivered_customer_date"] <=
        invalid_estimate["order_estimated_delivery_date"]
    ).sum()
)

Late deliveries: 465
Delivered on or before estimated date: 0


### Estimated Delivery Date Findings

- Identified 473 orders where carrier handoff occurred after the estimated delivery date.
- 465 orders had recorded customer delivery timestamps after the estimated date.
- The remaining 8 orders had missing customer delivery timestamps.
- All 8 incomplete records were associated with orders marked as shipped.
- These 8 orders cannot be classified as late or on-time deliveries.
- All 473 records will be retained for further analysis.

In [75]:
# Implement Quality Flags

# Missing timestamps

orders["missing_approval_date"] = (
    orders["order_approved_at"].isna()
)

orders["missing_carrier_date"] = (
    orders["order_delivered_carrier_date"].isna()
)

orders["missing_customer_delivery_date"] = (
    orders["order_delivered_customer_date"].isna()
)

In [84]:
# Carrier handoff occurred before approval
orders["invalid_carrier_sequence"] = (
    orders["order_approved_at"].notna() &
    orders["order_delivered_carrier_date"].notna() &
    (
        orders["order_delivered_carrier_date"] <
        orders["order_approved_at"]
    )
)

# Customer delivery occurred before carrier handoff
orders["invalid_customer_delivery_sequence"] = (
    orders["order_delivered_carrier_date"].notna() &
    orders["order_delivered_customer_date"].notna() &
    (
        orders["order_delivered_customer_date"] <
        orders["order_delivered_carrier_date"]
    )
)

# Approval occurred before purchase
orders["invalid_approval_sequence"] = (
    orders["order_approved_at"].notna() &
    (
        orders["order_approved_at"] <
        orders["order_purchase_timestamp"]
    )
)

# Customer received order after estimated delivery
orders["late_delivery"] = (
    orders["order_delivered_customer_date"].notna() &
    (
        orders["order_delivered_customer_date"] >
        orders["order_estimated_delivery_date"]
    )
)

# Carrier handoff occurred after estimated delivery
orders["carrier_after_estimated_date"] = (
    orders["order_delivered_carrier_date"].notna() &
    (
        orders["order_delivered_carrier_date"] >
        orders["order_estimated_delivery_date"]
    )
)

In [85]:
quality_flags = [
    "missing_approval_date",
    "missing_carrier_date",
    "missing_customer_delivery_date",
    "invalid_approval_sequence",
    "invalid_carrier_sequence",
    "invalid_customer_delivery_sequence"
]

orders["has_data_quality_issue"] = orders[quality_flags].any(axis=1)

In [86]:
data_quality_flags = [
    "missing_approval_date",
    "missing_carrier_date",
    "missing_customer_delivery_date",
    "invalid_approval_sequence",
    "invalid_carrier_sequence",
    "invalid_customer_delivery_sequence"
]

performance_flags = [
    "late_delivery",
    "carrier_after_estimated_date"
]

quality_summary = pd.DataFrame({
    "Affected Orders": orders[data_quality_flags].sum(),
    "Percentage": orders[data_quality_flags].mean() * 100
})

performance_summary = pd.DataFrame({
    "Affected Orders": orders[performance_flags].sum(),
    "Percentage": orders[performance_flags].mean() * 100
})

print("DATA QUALITY SUMMARY")
display(quality_summary.round(2))

print("DELIVERY PERFORMANCE SUMMARY")
display(performance_summary.round(2))

DATA QUALITY SUMMARY


,Affected Orders,Percentage
missing_approval_date,160,0.16
missing_carrier_date,1783,1.79
missing_customer_delivery_date,2965,2.98
invalid_approval_sequence,0,0.00
invalid_carrier_sequence,1359,1.37
invalid_customer_delivery_sequence,23,0.02


DELIVERY PERFORMANCE SUMMARY


,Affected Orders,Percentage
late_delivery,7827,7.87
carrier_after_estimated_date,473,0.48


In [82]:
print("Dataset Shape:", orders.shape)

print("\nData Types:")
print(orders.dtypes)

print("\nFirst 5 Records:")
orders.head()

Dataset Shape: (99441, 16)

Data Types:
order_id                                         str
customer_id                                      str
order_status                                     str
order_purchase_timestamp              datetime64[us]
order_approved_at                     datetime64[us]
order_delivered_carrier_date          datetime64[us]
order_delivered_customer_date         datetime64[us]
order_estimated_delivery_date         datetime64[us]
missing_approval_date                           bool
missing_carrier_date                            bool
missing_customer_delivery_date                  bool
invalid_carrier_sequence                        bool
invalid_customer_delivery_sequence              bool
late_delivery                                   bool
carrier_after_estimated_date                    bool
has_data_quality_issue                          bool
dtype: object

First 5 Records:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,missing_approval_date,missing_carrier_date,missing_customer_delivery_date,invalid_carrier_sequence,invalid_customer_delivery_sequence,late_delivery,carrier_after_estimated_date,has_data_quality_issue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,False,False,False,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,False,False,False,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,False,False,False,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,False,False,False,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,False,False,False,False,False


In [87]:
orders.columns = (
    orders.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'missing_approval_date', 'missing_carrier_date', 'missing_customer_delivery_date', 'invalid_carrier_sequence', 'invalid_customer_delivery_sequence', 'late_delivery', 'carrier_after_estimated_date', 'has_data_quality_issue', 'invalid_approval_sequence']


## Data Quality Flag Implementation

Created additional Boolean columns to identify missing timestamps, chronological inconsistencies, and delivery performance.

### Missing Timestamp Flags
- `missing_approval_date`
- `missing_carrier_date`
- `missing_customer_delivery_date`

### Chronological Integrity Flags
- `invalid_approval_sequence`
- `invalid_carrier_sequence`
- `invalid_customer_delivery_sequence`

### Delivery Performance Flags
- `late_delivery`
- `carrier_after_estimated_date`

### Overall Quality Flag
- `has_data_quality_issue`

### Cleaning Decisions
- Original order records were retained.
- Missing timestamps were not imputed because actual event times could not be established.
- Chronologically inconsistent timestamps were flagged rather than automatically corrected.
- Late deliveries were retained as valid business outcomes.
- Missing timestamps were evaluated in the context of order status.
- Individual quality flags will be used to determine which records are appropriate for specific analyses.

In [89]:
print("Missing Order IDs:", orders["order_id"].isna().sum())
print("Duplicate Order IDs:", orders["order_id"].duplicated().sum())
print("Unique Order IDs:", orders["order_id"].nunique())

Missing Order IDs: 0
Duplicate Order IDs: 0
Unique Order IDs: 99441


In [90]:
print("Missing Customer IDs:", orders["customer_id"].isna().sum())
print("Unique Customer IDs:", orders["customer_id"].nunique())

Missing Customer IDs: 0
Unique Customer IDs: 99441


In [91]:
print(
    "Customer IDs with leading/trailing whitespace:",
    (orders["customer_id"] != orders["customer_id"].str.strip()).sum()
)

print(
    "Order IDs with leading/trailing whitespace:",
    (orders["order_id"] != orders["order_id"].str.strip()).sum()
)

Customer IDs with leading/trailing whitespace: 0
Order IDs with leading/trailing whitespace: 0


In [93]:
# Export cleaned dataset

orders.to_csv(
    "../data/processed/olist_orders_cleaned.csv",
    index=False
)